# 🧠 تدريب نموذج RNN لتصنيف نية رسائل دعم العملاء (لهجة عربية عامية)

**جزء من مشروع: Arabic Dialect Conversational Support Agent**

هذا النوتبوك مخصص للعمل داخل **Google Colab** ويغطي:
1. توليد داتا سيت عربي عامي غير نظيف عبر LLM (بديل عن داتا سيت جاهز غير متوفر بجودة كافية)
2. تنظيف وتطبيع النصوص العامية
3. فحوصات جودة صارمة (Quality Gates)
4. تدريب **أكثر من نموذج RNN** ومقارنتها
5. تقييم شامل (classification report + confusion matrix)
6. حفظ الموديل النهائي والتوكنايزر بصيغة جاهزة للاستخدام مباشرة في الـ backend (Flask + LangGraph)

> ⚠️ **تنبيه مهم جداً**: دوال التطبيع (normalization) المستخدمة هنا **يجب أن تُنسخ حرفياً** إلى `backend/agent/normalizer.py` في مشروع الوكيل. أي اختلاف بسيط بين التطبيع هنا وهناك سيكسر دقة النموذج وقت الاستخدام الفعلي دون أي خطأ ظاهر (Silent Failure). راجع قسم 6.1 في ملف `ADR.md`.


## 1️⃣ الإعداد (Setup) وتثبيت المكتبات

In [ ]:
!pip install -q tensorflow scikit-learn pandas numpy matplotlib seaborn tqdm requests
print("✅ تم تثبيت المكتبات بنجاح")


In [ ]:
import os
import re
import json
import time
import pickle
import random
from getpass import getpass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from tqdm.auto import tqdm

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, GRU, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.utils.class_weight import compute_class_weight

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

print("✅ تم استيراد كل المكتبات")


In [ ]:
# 🔑 إعداد مفتاح الـ API (لا يُكتب المفتاح مطلقاً داخل الكود بشكل ثابت)
# الأولوية: Colab Secrets (userdata) -> متغير بيئة موجود مسبقاً -> إدخال يدوي آمن (getpass)

COMMANDCODE_API_KEY = None

try:
    from google.colab import userdata
    COMMANDCODE_API_KEY = userdata.get('COMMANDCODE_API_KEY')
    print("✅ تم جلب المفتاح من Colab Secrets")
except Exception:
    pass

if not COMMANDCODE_API_KEY:
    COMMANDCODE_API_KEY = os.environ.get("COMMANDCODE_API_KEY")

if not COMMANDCODE_API_KEY:
    COMMANDCODE_API_KEY = getpass("أدخل COMMANDCODE_API_KEY (لن يظهر أثناء الكتابة): ")

assert COMMANDCODE_API_KEY, "❌ لم يتم توفير مفتاح API — لا يمكن المتابعة بدونه"
print("✅ المفتاح جاهز للاستخدام")

LLM_MODEL = "deepseek/deepseek-v4.1-flash"  # يمكن تغييره لأي موديل متاح عبر COMMANDCODE
COMMANDCODE_URL = "https://api.commandcode.ai/provider/v1/chat/completions"



## 2️⃣ توليد الداتا سيت (Synthetic Arabic Dialect Dataset Generation)

**الاستراتيجية**: نولّد رسائل عملاء بلهجة عامية واقعية (خليجية/يمنية)، موزعة على 4 فئات نية:
- `complaint` — شكوى
- `order_inquiry` — استفسار عن طلب
- `return_request` — طلب استرجاع/استبدال
- `other` — أخرى (تحية، سؤال عام)

كل دفعة (batch) تولّد عدة رسائل متنوعة بأنماط شخصية مختلفة لتفادي التكرار الأسلوبي.


In [ ]:
PRODUCTS = [
    "سماعة بلوتوث", "شاحن سريع", "ساعة ذكية", "جوال سامسونج", "لابتوب ديل",
    "طقم مطبخ", "مكواة بخار", "مكنسة كهربائية", "كرسي مكتب", "طاولة قهوة",
    "حذاء رياضي", "شنطة ظهر", "نظارة شمسية", "عطر رجالي", "بطانية شتوية"
]

PERSONAS = [
    "عميل عصبي ومستعجل، يكتب بجمل قصيرة وقد يستخدم علامات تعجب كثيرة",
    "عميل مهذب جداً ويشرح المشكلة بتفصيل",
    "عميل مقتضب جداً، رسالته سطر واحد بس",
    "عميل يستخدم رموز تعبيرية (إيموجي) بشكل طبيعي",
    "عميل يخلط أحياناً كلمات إنجليزية بسيطة وسط كلامه العربي (زي 'order' و 'delivery')",
    "عميل كبير بالسن يكتب بأسلوب تقليدي أكثر رسمية لكن بالعامية"
]

CATEGORIES = {
    "complaint": "شكوى من منتج أو خدمة (مثلاً: المنتج وصل تالف، الخدمة سيئة، تأخير غير مبرر) بدون سؤال محدد عن حالة طلب",
    "order_inquiry": "سؤال محدد عن حالة أو موعد وصول طلب موجود لديه بالفعل",
    "return_request": "رغبة صريحة بإرجاع المنتج أو استبداله",
    "other": "رسالة عامة غير متعلقة بطلب محدد: تحية، سؤال عن منتج قبل الشراء، شكر، سؤال عن سياسة عامة",
}

GENERATION_SYSTEM_PROMPT = """أنت مولّد بيانات تدريب لنموذج NLP. مهمتك إنشاء رسائل واتساب واقعية
يكتبها عملاء عرب (لهجة خليجية/يمنية عامية) لمتجر إلكتروني، لغرض تدريب نموذج تصنيف نصوص.

قواعد صارمة يجب اتباعها:
1. اكتب باللهجة العامية الحقيقية فقط، ممنوع الفصحى الرسمية.
2. أدخل أخطاء إملائية طبيعية كما يكتبها الناس فعلاً على الجوال (حذف همزات، دمج كلمات، أخطاء شائعة).
3. نوّع طول الرسائل (بعضها سطر واحد، بعضها عدة أسطر).
4. لا تكرر نفس الصياغة أو نفس الجمل الافتتاحية بين الرسائل.
5. أرجع الإخراج **بصيغة JSON فقط** بدون أي نص إضافي قبله أو بعده، على شكل:
{"messages": ["رسالة 1", "رسالة 2", ...]}
"""

def build_user_prompt(category_key, category_desc, persona, product, n=5):
    return f"""المطلوب: ولّد {n} رسائل عميل مختلفة تماماً عن بعضها.
الفئة المطلوبة: {category_desc}
شخصية الكاتب المطلوبة لهذه الدفعة: {persona}
المنتج المذكور (اذكره بصيغة طبيعية داخل بعض الرسائل، وليس بالضرورة كلها): {product}
تذكر: أرجع JSON فقط بالصيغة المحددة، بدون أي شرح إضافي."""

print(f"✅ عدد المنتجات: {len(PRODUCTS)} | عدد الفئات: {len(CATEGORIES)} | عدد الأنماط الشخصية: {len(PERSONAS)}")


In [ ]:
def call_llm(system_prompt, user_prompt, max_retries=3):
    """استدعاء CommandCode API مع إعادة محاولة عند الفشل."""
    headers = {
        "Authorization": f"Bearer {COMMANDCODE_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": LLM_MODEL,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        "temperature": 1.0,
    }
    for attempt in range(max_retries):
        try:
            resp = requests.post(COMMANDCODE_URL, headers=headers, json=payload, timeout=60)
            resp.raise_for_status()
            content = resp.json()["choices"][0]["message"]["content"]
            # تنظيف أي fences مثل ```json قبل الـ parsing
            content = re.sub(r"^```json|```$", "", content.strip(), flags=re.MULTILINE).strip()
            data = json.loads(content)
            return data.get("messages", [])
        except Exception as e:
            print(f"  ⚠️ محاولة {attempt+1} فشلت: {e}")
            time.sleep(2 * (attempt + 1))
    return []

In [ ]:
# ⚙️ إعدادات التوليد — عدّل TARGET_PER_CATEGORY حسب الحجم المطلوب
# اقتراح ADR: 4000-6000 صف إجمالي => ~1000-1500 لكل فئة من أصل 4 فئات
BATCH_SIZE = 5
TARGET_PER_CATEGORY = 1000   # عدّلها حسب الوقت والميزانية المتاحة لاستدعاءات الـ API

generated_rows = []

for cat_key, cat_desc in CATEGORIES.items():
    print(f"\n🔄 توليد فئة: {cat_key} ({cat_desc[:40]}...)")
    count = 0
    pbar = tqdm(total=TARGET_PER_CATEGORY)
    while count < TARGET_PER_CATEGORY:
        persona = random.choice(PERSONAS)
        product = random.choice(PRODUCTS)
        order_id = random.randint(1000, 9999)
        user_prompt = build_user_prompt(cat_key, cat_desc, persona, product, n=BATCH_SIZE)
        messages = call_llm(GENERATION_SYSTEM_PROMPT, user_prompt)
        for m in messages:
            generated_rows.append({
                "text": m,
                "category": cat_key,
                "product_name": product,
                "order_id": order_id,
            })
            count += 1
            pbar.update(1)
            if count >= TARGET_PER_CATEGORY:
                break
        time.sleep(0.5)  # تفادي rate limiting
    pbar.close()

df_raw = pd.DataFrame(generated_rows)
print(f"\n✅ إجمالي الصفوف المولّدة: {len(df_raw)}")
df_raw.to_csv("raw_dataset.csv", index=False, encoding="utf-8-sig")
df_raw.head(10)


In [ ]:
from google.colab import files

# لتحميل الداتا سيت الكاملة
files.download('raw_dataset.csv')

# لتحميل عينة المراجعة (اختياري)
files.download('manual_review_sample.csv')

## 3️⃣ التنظيف والتطبيع (Cleaning & Normalization)\n\n⚠️ **هذه الدوال حرجة — انسخها حرفياً لملف `backend/agent/normalizer.py`**

In [ ]:
def normalize_arabic(text: str) -> str:
    """تطبيع النص العربي العامي: توحيد الهمزات، التاء المربوطة، الألف المقصورة،
    إزالة التشكيل، إزالة تكرار الحروف الزائد، توحيد الأرقام.
    """
    if not isinstance(text, str):
        return ""

    text = text.strip()

    # إزالة التشكيل (diacritics)
    arabic_diacritics = re.compile(r'[\u0617-\u061A\u064B-\u0652]')
    text = arabic_diacritics.sub('', text)

    # توحيد الهمزات
    text = re.sub(r'[إأآا]', 'ا', text)
    # توحيد التاء المربوطة والهاء
    text = re.sub(r'ة', 'ه', text)
    # توحيد الألف المقصورة والياء
    text = re.sub(r'ى', 'ي', text)
    # توحيد الأرقام العربية-الهندية إلى أرقام إنجليزية
    arabic_indic = "٠١٢٣٤٥٦٧٨٩"
    western = "0123456789"
    text = text.translate(str.maketrans(arabic_indic, western))
    # إزالة تكرار الحروف الزائد (مثال: "ابددد" -> "ابد")
    text = re.sub(r'(.)\1{2,}', r'\1', text)
    # إزالة المسافات الزائدة
    text = re.sub(r'\s+', ' ', text).strip()

    return text


def inject_extra_noise(text: str, noise_prob: float = 0.05) -> str:
    """حقن ضوضاء طبيعية إضافية برمجياً (اختياري، يُستخدم على نسخة موازية للتنويع)."""
    chars = list(text)
    result = []
    for c in chars:
        if random.random() < noise_prob and c.isalpha():
            continue  # حذف حرف عشوائي بنسبة صغيرة
        result.append(c)
    return "".join(result)


# تطبيق التطبيع على الداتا سيت
df_raw["text_normalized"] = df_raw["text"].apply(normalize_arabic)

# إزالة الصفوف الفارغة بعد التطبيع
df_raw = df_raw[df_raw["text_normalized"].str.len() > 2].reset_index(drop=True)

print(f"✅ عدد الصفوف بعد التنظيف: {len(df_raw)}")
df_raw[["text", "text_normalized", "category"]].sample(5)


## 4️⃣ فحوصات الجودة (Quality Gates) — إلزامية حسب `ADR.md` قسم 5.4

In [ ]:
print("📋 تقرير فحوصات الجودة\n" + "="*50)

# 1) توزيع الفئات
dist = df_raw["category"].value_counts(normalize=True) * 100
print("\n1) توزيع الفئات (%):")
print(dist)
min_pct = dist.min()
gate1 = min_pct >= 15
print(f"   {'✅' if gate1 else '❌'} أقل نسبة فئة = {min_pct:.1f}% (يجب >= 15%)")

# 2) نسبة التكرار الحرفي الكامل
dup_pct = (df_raw.duplicated(subset=["text_normalized"]).sum() / len(df_raw)) * 100
gate2 = dup_pct <= 1.0
print(f"\n2) {'✅' if gate2 else '❌'} نسبة التكرار الحرفي = {dup_pct:.2f}% (يجب <= 1%)")

# 3) توزيع طول النصوص
df_raw["text_len"] = df_raw["text_normalized"].apply(lambda x: len(x.split()))
print(f"\n3) متوسط طول الرسالة (كلمات): {df_raw['text_len'].mean():.1f} | الوسيط: {df_raw['text_len'].median():.0f}")

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
sns.countplot(data=df_raw, x="category")
plt.title("توزيع الفئات")
plt.subplot(1, 2, 2)
sns.histplot(df_raw["text_len"], bins=30)
plt.title("توزيع طول الرسائل (بالكلمات)")
plt.tight_layout()
plt.savefig("dataset_summary_plots.png", dpi=120)
plt.show()

# حفظ تقرير الجودة كملف نصي
with open("dataset_summary.md", "w", encoding="utf-8") as f:
    f.write("# تقرير جودة الداتا سيت\n\n")
    f.write(f"- إجمالي الصفوف: {len(df_raw)}\n")
    f.write(f"- توزيع الفئات:\n{dist.to_string()}\n\n")
    f.write(f"- نسبة التكرار: {dup_pct:.2f}%\n")
    f.write(f"- متوسط طول الرسالة: {df_raw['text_len'].mean():.1f} كلمة\n")
    f.write(f"- Gate 1 (توازن الفئات): {'PASS' if gate1 else 'FAIL'}\n")
    f.write(f"- Gate 2 (نسبة التكرار): {'PASS' if gate2 else 'FAIL'}\n")

print("\n✅ تم حفظ dataset_summary.md و dataset_summary_plots.png")
assert gate1 and gate2, "❌ الداتا سيت لم تجتز فحوصات الجودة — راجع الإعدادات أعلاه قبل المتابعة"


In [ ]:
# 📝 مراجعة يدوية إلزامية (Manual Review Sample) — حسب ADR: 100 صف على الأقل
manual_review_sample = df_raw.sample(100, random_state=RANDOM_SEED)[["text", "text_normalized", "category"]]
manual_review_sample.to_csv("manual_review_sample.csv", index=False, encoding="utf-8-sig")
print("✅ تم تصدير manual_review_sample.csv — يجب مراجعتها يدوياً وتسجيل نسبة الأخطاء في dataset_summary.md قبل اعتماد الداتا سيت نهائياً")


## 5️⃣ التحضير للتدريب (Train/Val/Test Split + Tokenization)

In [ ]:
# ترميز الفئات
label_encoder = LabelEncoder()
df_raw["label"] = label_encoder.fit_transform(df_raw["category"])
NUM_CLASSES = len(label_encoder.classes_)
print("الفئات:", list(label_encoder.classes_))

# تقسيم البيانات (Stratified)
X_train_text, X_temp_text, y_train, y_temp = train_test_split(
    df_raw["text_normalized"].values, df_raw["label"].values,
    test_size=0.3, stratify=df_raw["label"].values, random_state=RANDOM_SEED
)
X_val_text, X_test_text, y_val, y_test = train_test_split(
    X_temp_text, y_temp, test_size=0.5, stratify=y_temp, random_state=RANDOM_SEED
)

print(f"Train: {len(X_train_text)} | Val: {len(X_val_text)} | Test: {len(X_test_text)}")

# بناء التوكنايزر من بيانات التدريب فقط (تفادي data leakage)
MAX_VOCAB = 15000
MAX_LEN = 40

tokenizer = Tokenizer(num_words=MAX_VOCAB, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train_text)

X_train = pad_sequences(tokenizer.texts_to_sequences(X_train_text), maxlen=MAX_LEN, padding="post", truncating="post")
X_val = pad_sequences(tokenizer.texts_to_sequences(X_val_text), maxlen=MAX_LEN, padding="post", truncating="post")
X_test = pad_sequences(tokenizer.texts_to_sequences(X_test_text), maxlen=MAX_LEN, padding="post", truncating="post")

y_train_cat = to_categorical(y_train, NUM_CLASSES)
y_val_cat = to_categorical(y_val, NUM_CLASSES)
y_test_cat = to_categorical(y_test, NUM_CLASSES)

vocab_size = min(MAX_VOCAB, len(tokenizer.word_index) + 1)
print(f"✅ حجم المفردات الفعلي: {vocab_size}")

# أوزان الفئات (في حال عدم توازن كامل)
class_weights_arr = compute_class_weight(class_weight="balanced", classes=np.unique(y_train), y=y_train)
class_weights = dict(enumerate(class_weights_arr))
print("أوزان الفئات:", class_weights)


## 6️⃣ تعريف النماذج (نُدرّب 3 معماريات ونقارن بينها)

In [ ]:
EMBED_DIM = 128

def build_bilstm_model():
    model = Sequential([
        Embedding(vocab_size, EMBED_DIM, input_length=MAX_LEN),
        Bidirectional(LSTM(64)),
        Dropout(0.4),
        Dense(64, activation="relu"),
        Dropout(0.3),
        Dense(NUM_CLASSES, activation="softmax"),
    ], name="BiLSTM_Baseline")
    model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
    return model

def build_stacked_bilstm_model():
    model = Sequential([
        Embedding(vocab_size, EMBED_DIM, input_length=MAX_LEN),
        Bidirectional(LSTM(64, return_sequences=True)),
        Dropout(0.4),
        Bidirectional(LSTM(32)),
        Dropout(0.3),
        Dense(64, activation="relu"),
        Dense(NUM_CLASSES, activation="softmax"),
    ], name="Stacked_BiLSTM")
    model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
    return model

def build_gru_model():
    model = Sequential([
        Embedding(vocab_size, EMBED_DIM, input_length=MAX_LEN),
        Bidirectional(GRU(64)),
        Dropout(0.4),
        Dense(64, activation="relu"),
        Dropout(0.3),
        Dense(NUM_CLASSES, activation="softmax"),
    ], name="BiGRU")
    model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
    return model

candidate_models = {
    "BiLSTM_Baseline": build_bilstm_model(),
    "Stacked_BiLSTM": build_stacked_bilstm_model(),
    "BiGRU": build_gru_model(),
}

for name, m in candidate_models.items():
    print(f"\n{'='*20} {name} {'='*20}")
    m.summary()


## 7️⃣ التدريب المكثف لكل نموذج (Early Stopping + Checkpointing + LR Scheduling)

In [ ]:
EPOCHS = 60  # تدريب مكثف — EarlyStopping سيوقف تلقائياً عند التوقف عن التحسن
BATCH = 32

training_histories = {}
os.makedirs("checkpoints", exist_ok=True)

for name, model in candidate_models.items():
    print(f"\n🚀 بدء تدريب: {name}")
    callbacks = [
        EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True, verbose=1),
        ModelCheckpoint(f"checkpoints/{name}.keras", monitor="val_accuracy", save_best_only=True, verbose=0),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-6, verbose=1),
    ]
    history = model.fit(
        X_train, y_train_cat,
        validation_data=(X_val, y_val_cat),
        epochs=EPOCHS,
        batch_size=BATCH,
        class_weight=class_weights,
        callbacks=callbacks,
        verbose=2,
    )
    training_histories[name] = history.history
    print(f"✅ انتهى تدريب {name}")


In [ ]:
# 📈 رسم منحنيات التدريب لكل النماذج
fig, axes = plt.subplots(len(candidate_models), 2, figsize=(12, 4 * len(candidate_models)))
for i, (name, hist) in enumerate(training_histories.items()):
    axes[i, 0].plot(hist["accuracy"], label="train")
    axes[i, 0].plot(hist["val_accuracy"], label="val")
    axes[i, 0].set_title(f"{name} — Accuracy")
    axes[i, 0].legend()

    axes[i, 1].plot(hist["loss"], label="train")
    axes[i, 1].plot(hist["val_loss"], label="val")
    axes[i, 1].set_title(f"{name} — Loss")
    axes[i, 1].legend()

plt.tight_layout()
plt.savefig("training_curves.png", dpi=120)
plt.show()


## 8️⃣ التقييم الشامل والمقارنة بين النماذج (Test Set)

In [ ]:
comparison_rows = []
class_names = list(label_encoder.classes_)

for name, model in candidate_models.items():
    print(f"\n{'='*25} تقييم: {name} {'='*25}")
    y_pred_probs = model.predict(X_test, verbose=0)
    y_pred = np.argmax(y_pred_probs, axis=1)

    report = classification_report(y_test, y_pred, target_names=class_names, output_dict=True)
    print(classification_report(y_test, y_pred, target_names=class_names))

    macro_f1 = report["macro avg"]["f1-score"]
    accuracy = report["accuracy"]
    comparison_rows.append({"model": name, "test_accuracy": accuracy, "macro_f1": macro_f1})

    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
    plt.title(f"Confusion Matrix — {name}")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.savefig(f"confusion_matrix_{name}.png", dpi=120)
    plt.show()

comparison_df = pd.DataFrame(comparison_rows).sort_values("macro_f1", ascending=False)
print("\n📊 جدول المقارنة النهائي:")
print(comparison_df.to_string(index=False))
comparison_df.to_csv("model_comparison.csv", index=False)

BEST_MODEL_NAME = comparison_df.iloc[0]["model"]
print(f"\n🏆 أفضل نموذج: {BEST_MODEL_NAME}")


## 9️⃣ حفظ الموديل النهائي والأدوات المصاحبة (جاهز للنقل إلى الـ Backend)

In [ ]:
os.makedirs("final_artifacts", exist_ok=True)

best_model = candidate_models[BEST_MODEL_NAME]
best_model.save("final_artifacts/model.keras")

with open("final_artifacts/tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

with open("final_artifacts/label_encoder.pkl", "wb") as f:
    pickle.dump(label_encoder, f)

# حفظ إعدادات مهمة يحتاجها الـ backend (MAX_LEN وغيره)
with open("final_artifacts/config.json", "w", encoding="utf-8") as f:
    json.dump({
        "max_len": MAX_LEN,
        "vocab_size": vocab_size,
        "classes": class_names,
        "best_model": BEST_MODEL_NAME,
    }, f, ensure_ascii=False, indent=2)

print("✅ تم حفظ: model.keras, tokenizer.pkl, label_encoder.pkl, config.json داخل final_artifacts/")
print("📌 الخطوة التالية: انسخ هذا المجلد بالكامل إلى backend/model/artifacts/ في مشروع الوكيل")


## 🔟 اختبار سلامة نهائي (Sanity Check) — تحميل الملفات من الصفر والتنبؤ

In [ ]:
# نحاكي هنا بالضبط ما سيفعله backend/model/rnn_classifier.py عند التشغيل الفعلي

loaded_model = load_model("final_artifacts/model.keras")
with open("final_artifacts/tokenizer.pkl", "rb") as f:
    loaded_tokenizer = pickle.load(f)
with open("final_artifacts/label_encoder.pkl", "rb") as f:
    loaded_label_encoder = pickle.load(f)
with open("final_artifacts/config.json", "r", encoding="utf-8") as f:
    loaded_config = json.load(f)

sample_messages = [
    "وينه طلبي يا اخي تاخر كثير مب طبيعي",
    "ابغى استرجع المنتج لان مو مطابق للمواصفات",
    "هلا، عندكم توصيل لمنطقة الرياض؟",
]

print("🔍 نتائج الاختبار على جمل تجريبية:\n")
for msg in sample_messages:
    norm = normalize_arabic(msg)
    seq = pad_sequences(loaded_tokenizer.texts_to_sequences([norm]), maxlen=loaded_config["max_len"], padding="post")
    pred_probs = loaded_model.predict(seq, verbose=0)[0]
    pred_label = loaded_label_encoder.inverse_transform([np.argmax(pred_probs)])[0]
    print(f"الرسالة: {msg}")
    print(f"  → التصنيف المتوقع: {pred_label} (ثقة: {pred_probs.max():.2f})\n")

print("✅ لو الفئات المتوقعة أعلاه منطقية، فالـ pipeline (تطبيع + tokenizer + model) متسق وجاهز للنقل للمشروع")
print("⚠️ لو النتائج غير منطقية، لا تنقل الملفات للمشروع — راجع تطابق دالة normalize_arabic مرة أخرى")


## ✅ ملخص المخرجات النهائية لهذا النوتبوك

| الملف | الوجهة في المشروع الرئيسي |
|---|---|
| `final_artifacts/model.keras` | `backend/model/artifacts/model.keras` |
| `final_artifacts/tokenizer.pkl` | `backend/model/artifacts/tokenizer.pkl` |
| `final_artifacts/label_encoder.pkl` | `backend/model/artifacts/label_encoder.pkl` |
| `final_artifacts/config.json` | `backend/model/artifacts/config.json` |
| `dataset_summary.md` | `training/reports/dataset_summary.md` |
| `model_comparison.csv` | `training/reports/model_comparison.md` (يُحوَّل لجدول بالتقرير) |
| `confusion_matrix_*.png`, `training_curves.png` | `training/reports/` |

**تذكير أخير:** دالة `normalize_arabic()` في الخلية أعلاه يجب أن تُنسخ **حرفياً بدون أي تعديل** إلى `backend/agent/normalizer.py`.
